# Entrenamiento urbano en Google Colab Free

Notebook de preparación y ejecución controlada para la rama `urban-models-and-benchmark`. No descarga datasets automáticamente ni inicia entrenamientos al abrirlo.

**Importante:** Colab Free no garantiza GPU ni persistencia de la máquina. Los checkpoints se guardan en Google Drive; los datasets deben estar disponibles en el almacenamiento remoto de Colab mediante una fuente autorizada. Nunca descargues datasets al teléfono.


## 1. Configuración inicial
Ajusta la rama y las rutas si fuera necesario. Los datasets se ubican en el disco temporal de Colab (`/content/datasets`) para evitar lecturas lentas desde Drive. Los resultados se guardan en Drive.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json

REPO_URL = "https://github.com/Leohf04/Procesamiento-de-Im-genes-.git"
BRANCH = "urban-models-and-benchmark"
WORKDIR = Path("/content/Procesamiento-de-Im-genes-")
DATA_DIR = Path("/content/datasets")  # almacenamiento temporal remoto; no es el teléfono
DRIVE_RESULTS = Path("/content/drive/MyDrive/ProcesamientoImagenes/resultados")
CITYSCAPES_ROOT = DATA_DIR / "cityscapes"
BDD_ROOT = DATA_DIR / "bdd100k"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Configuración preparada")
print("Repositorio:", REPO_URL, "\nRama:", BRANCH)
print("Datos temporales:", DATA_DIR)
print("Resultados persistentes:", DRIVE_RESULTS)


## 2. Montar Google Drive
Autoriza el montaje desde la ventana de Google. Drive conserva resultados, no se usa como carpeta de entrenamiento de imágenes masivas.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for folder in ["checkpoints", "metricas", "graficas", "informes", "logs"]:
    (DRIVE_RESULTS / folder).mkdir(parents=True, exist_ok=True)
print("Drive listo:", DRIVE_RESULTS)


## 3. Clonar el repositorio privado
Crea en **Colab → Secrets** un secreto llamado `GITHUB_TOKEN` con un Fine-grained Personal Access Token de GitHub limitado a este repositorio y con permiso **Contents: Read-only**. No escribas el token en una celda ni lo compartas en el chat. Si no está en Secrets, el notebook solicitará el token de forma oculta durante esta sesión.


In [ ]:
from getpass import getpass
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if not github_token:
    github_token = getpass("GitHub token (no se mostrará): ")
if not github_token:
    raise RuntimeError("Se necesita un token GitHub de solo lectura para clonar el repositorio privado.")

askpass = Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "COLAB_GITHUB_TOKEN": github_token})
if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", BRANCH], check=True, env=env)
    subprocess.run(["git", "-C", str(WORKDIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", BRANCH], check=True, env=env)
else:
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(WORKDIR)], check=True, env=env)
del github_token, env
print(subprocess.check_output(["git", "-C", str(WORKDIR), "log", "-1", "--oneline"], text=True))


## 4. Dependencias compatibles con Colab
No reinstalamos `torch` ni `torchvision`: Colab ya proporciona builds preparados para su entorno CUDA. Instalamos únicamente dependencias adicionales del proyecto.


In [ ]:
%cd /content/Procesamiento-de-Im-genes-
!python -m pip install -q "opencv-python-headless>=4.8,<5" "scikit-learn>=1.3" "matplotlib>=3.7" "PyYAML>=6" "tqdm>=4.66" "joblib>=1.3" "onnx>=1.14" "onnxruntime>=1.17" "Pillow>=9" "pycocotools>=2.0.7"


## 5. Diagnóstico del entorno
Si `torchvision` no importa o no coincide con PyTorch, detén el proceso y comparte el error; no actualices CUDA/PyTorch a ciegas.


In [ ]:
import torch, torchvision
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    !nvidia-smi
else:
    print("AVISO: esta sesión no recibió GPU. No inicies entrenamiento pesado.")


In [ ]:
import subprocess
subprocess.run([sys.executable, "scripts/verify_setup.py"], check=True)


## 6. Comprobar datasets (sin descargarlos)
Cityscapes necesita `leftImg8bit` y `gtFine` de train/val. BDD100K carriles requiere imágenes y JSON de anotaciones, además de máscaras rasterizadas generadas por el script del proyecto. Esta celda solo verifica estructura; no intenta descargar nada.


In [ ]:
def count_cityscapes(root):
    root = Path(root)
    result = {}
    for split in ("train", "val"):
        images = list((root / "leftImg8bit" / split).glob("*/*_leftImg8bit.png"))
        labels = list((root / "gtFine" / split).glob("*/*_gtFine_labelIds.png"))
        result[split] = {"images": len(images), "semantic_labels": len(labels)}
    return result

if CITYSCAPES_ROOT.exists():
    print(json.dumps(count_cityscapes(CITYSCAPES_ROOT), indent=2))
else:
    print("Cityscapes no está en", CITYSCAPES_ROOT, "— pendiente de acceso/descarga autorizada directa en la nube.")
print("BDD100K root exists:", BDD_ROOT.exists())


## 7. Configurar entrenamiento
No ejecutes esta sección hasta tener un dataset autorizado, completo y validado. El valor `RUN_TRAINING` permanece desactivado deliberadamente. Los scripts ahora guardan un checkpoint reanudable `.last.pt` y aceptan `--resume`.

Para reanudar una ejecución interrumpida, conserva la misma ruta de checkpoint y añade `--resume`. `--epochs` representa el total final de épocas, no las épocas adicionales.


In [ ]:
RUN_TRAINING = False  # Cambiar a True solo después de validar dataset y GPU
TASK = "cityscapes"  # "cityscapes" o "lanes"
EPOCHS = 30
BATCH_SIZE = 2  # comenzar conservador en GPU gratuita
WORKERS = 2
RESUME = False

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("No hay GPU disponible en esta sesión; se cancela el entrenamiento.")
    if TASK == "cityscapes":
        if not (CITYSCAPES_ROOT / "leftImg8bit" / "train").exists():
            raise FileNotFoundError("Falta Cityscapes train en " + str(CITYSCAPES_ROOT))
        output = DRIVE_RESULTS / "checkpoints" / "deeplabv3plus_cityscapes.pt"
        command = [sys.executable, "scripts/train_urban_segmentation.py", "--task", "cityscapes",
                   "--data-root", str(CITYSCAPES_ROOT), "--epochs", str(EPOCHS),
                   "--batch-size", str(BATCH_SIZE), "--workers", str(WORKERS), "--output", str(output)]
    else:
        raise NotImplementedError("La preparación BDD100K debe validarse y configurarse por separado antes de entrenar.")
    if RESUME:
        command.append("--resume")
    subprocess.run(command, check=True)
else:
    print("Entrenamiento desactivado. Primero valida el dataset y la GPU.")


## 8. Evaluación y persistencia
Después de entrenar, ejecutaremos los evaluadores existentes y guardaremos métricas e informes en `DRIVE_RESULTS`. No subas datasets ni checkpoints grandes al repositorio GitHub. Los artefactos grandes permanecen en Drive; el código y reportes ligeros permanecen en GitHub.

**Al terminar cada sesión:** confirma que los archivos `.pt`, `.last.pt` y `.history.json` existen en Drive antes de desconectar. Si la sesión se interrumpe, monta Drive, clona/actualiza el repo y reanuda con `--resume`.
